In [16]:
#للتحديث المستمر عند اي تعديل 
%load_ext autoreload
%autoreload 2

#لاضافة مسار مجلد المشروع 
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath("G:\Programming\Computer Vision\Smart-humanitarian-aid-distribution-system-project"))



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
from src.preprocessing.load_data import load_data

df = load_data(r"G:\\Programming\\Computer Vision\\Smart-humanitarian-aid-distribution-system-project\\data\\raw\\families_200_housing_randomized.xlsx")

df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   FamilyID       200 non-null    int64 
 1   SpecialCase    200 non-null    int64 
 2   Income         200 non-null    int64 
 3   FamilyMembers  200 non-null    int64 
 4   Housing        200 non-null    object
 5   Region         200 non-null    object
dtypes: int64(4), object(2)
memory usage: 9.5+ KB


In [18]:

df.head()


,FamilyID,SpecialCase,Income,FamilyMembers,Housing,Region
0,96,0,1120,3,مدمر جزئيا,Center
1,16,0,1156,4,مدمر بالكامل,North
2,31,0,659,2,مدمر بالكامل,North
3,159,0,1864,2,مخيم ايواء/خيمة,Center
4,129,0,2860,3,ايجار,Center


In [19]:

# for watching a missing value in a columns

from src.preprocessing.clean_data import clean_data


df = clean_data(df)
df.isnull().sum()

FamilyID         0
SpecialCase      0
Income           0
FamilyMembers    0
Housing          0
Region           0
dtype: int64

In [20]:
from src.features.feature_engineering import feature_engineering

df = feature_engineering(df)

df.head()
df.columns

Index(['FamilyID', 'SpecialCase', 'Income', 'FamilyMembers', 'Housing',
       'Region_East', 'Region_North', 'Region_South', 'Region_West'],
      dtype='object')

In [21]:
df[['SpecialCase','Income','FamilyMembers','Housing']].isnull().sum()

SpecialCase      0
Income           0
FamilyMembers    0
Housing          0
dtype: int64

In [22]:
print(df[['SpecialCase', 'Income', 'FamilyMembers', 'Housing']].dtypes)

SpecialCase        int64
Income           float64
FamilyMembers    float64
Housing          float64
dtype: object


In [23]:
print(df['Housing'].unique())

[0.7 1.  0.2 0.4]


In [24]:
print(df[['SpecialCase', 'Income', 'FamilyMembers', 'Housing']].head(10))

   SpecialCase  Income  FamilyMembers  Housing
0            0     0.5            0.5      0.7
1            0     0.5            0.5      1.0
2            0     1.0            0.2      1.0
3            0     0.5            0.2      0.2
4            0     0.0            0.5      0.4
5            1     0.5            0.2      0.4
6            1     1.0            0.5      0.7
7            0     1.0            0.2      0.2
8            0     0.5            0.5      0.2
9            0     0.5            0.2      1.0


In [25]:
from src.scoring.priority_score import calculate_priority_score

df = calculate_priority_score(df)

df[['PriorityScore', 'PriorityLevel']]

,PriorityScore,PriorityLevel
0,0.32,Low
1,0.35,Low
2,0.44,Low
3,0.21,Low
4,0.14,Low
...,...,...
195,0.29,Low
196,0.54,Medium
197,0.87,High
198,0.16,Low


In [26]:
print("Rows:", len(df))
print("Missing values:")
print(df[["SpecialCase", "Income", "FamilyMembers", "Housing",
          "PriorityScore", "PriorityLevel"]].isna().sum())

print("\nPriority levels:")
print(df["PriorityLevel"].value_counts())

print("\nScore range:")
print(df["PriorityScore"].min(), "→", df["PriorityScore"].max())

Rows: 200
Missing values:
SpecialCase      0
Income           0
FamilyMembers    0
Housing          0
PriorityScore    0
PriorityLevel    0
dtype: int64

Priority levels:
PriorityLevel
Low       132
Medium     51
High       17
Name: count, dtype: int64

Score range:
0.06000000000000001 → 0.9099999999999999


In [27]:
test = pd.DataFrame([{
    "SpecialCase": 1,
    "Income": 1.0,
    "FamilyMembers": 1.0,
    "Housing": 1.0
}])

test = calculate_priority_score(test)

print(test[["PriorityScore", "PriorityLevel"]])

   PriorityScore PriorityLevel
0            1.0          High


In [28]:
from src.models.predict import predict_priority

test_data = {
    "SpecialCase": 1,
    "Income": 1.0,
    "FamilyMembers": 1.0,
    "Housing": 1.0
}

result = predict_priority(test_data)

print(result)

{'PriorityScore': 0.84, 'PriorityLevel': 'High'}


In [29]:
# الجدول الناتج 
df.head()

,FamilyID,SpecialCase,Income,FamilyMembers,Housing,Region_East,Region_North,Region_South,Region_West,PriorityScore,PriorityLevel
0,96,0,0.5,0.5,0.7,False,False,False,False,0.32,Low
1,16,0,0.5,0.5,1.0,False,True,False,False,0.35,Low
2,31,0,1.0,0.2,1.0,False,True,False,False,0.44,Low
3,159,0,0.5,0.2,0.2,False,False,False,False,0.21,Low
4,129,0,0.0,0.5,0.4,False,False,False,False,0.14,Low


In [30]:
# الجدول الناتج بعد ترتيبهم حسب الاولوية
df = df.sort_values(by='PriorityScore', ascending=False)
df.head()

,FamilyID,SpecialCase,Income,FamilyMembers,Housing,Region_East,Region_North,Region_South,Region_West,PriorityScore,PriorityLevel
189,75,1,1.0,0.7,0.7,False,False,False,False,0.91,High
30,10,1,1.0,0.5,1.0,False,False,True,False,0.90,High
134,44,1,1.0,0.5,1.0,False,False,False,True,0.90,High
180,2,1,1.0,0.5,1.0,False,True,False,False,0.90,High
37,138,1,1.0,0.7,0.4,False,False,True,False,0.88,High


In [31]:
#اذا بدي اعلى 50 بالاولوية   
df = df.sort_values(by='PriorityScore', ascending=False)
top_50 = df.head(50)

top_50

,FamilyID,SpecialCase,Income,FamilyMembers,Housing,Region_East,Region_North,Region_South,Region_West,PriorityScore,PriorityLevel
189,75,1,1.0,0.7,0.7,False,False,False,False,0.91,High
30,10,1,1.0,0.5,1.0,False,False,True,False,0.90,High
134,44,1,1.0,0.5,1.0,False,False,False,True,0.90,High
180,2,1,1.0,0.5,1.0,False,True,False,False,0.90,High
37,138,1,1.0,0.7,0.4,False,False,True,False,0.88,High
171,89,1,1.0,0.5,0.7,True,False,False,False,0.87,High
6,70,1,1.0,0.5,0.7,False,True,False,False,0.87,High
197,93,1,1.0,0.5,0.7,True,False,False,False,0.87,High
161,135,1,1.0,0.5,0.4,False,False,True,False,0.84,High
187,117,1,1.0,0.5,0.4,True,False,False,False,0.84,High


In [32]:
#اذا بدي اعلى 50 بالاولوية مع عرض بعض الاعمدة المهمة
df = df.sort_values(by='PriorityScore', ascending=False)
top_50 = df.head(50)
top_50[['FamilyID', 'PriorityScore', 'PriorityLevel']]

,FamilyID,PriorityScore,PriorityLevel
189,75,0.91,High
30,10,0.90,High
134,44,0.90,High
180,2,0.90,High
37,138,0.88,High
171,89,0.87,High
6,70,0.87,High
197,93,0.87,High
161,135,0.84,High
187,117,0.84,High


In [33]:

import os

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

save_path = os.path.join(BASE_DIR, "data", "processed")

os.makedirs(save_path, exist_ok=True)

top_50.to_csv(os.path.join(save_path, "top_50.csv"), index=False)

# CSV   حفظت أعلى 50 عائلات   في مجلد البيانات المعالجة في ملف  
# النقطتين في المسار تعني ارجع مجلد واحد للخلف حتى يحفظ في مجلد المشروع وليس داخل مجلد notebooke

In [34]:


X = df[
    [
        "SpecialCase",
        "Income",
        "FamilyMembers",
        "Housing"
    ]
]

y = df["PriorityScore"]

X = X.astype(float)

X (Features) → المعلومات التي نُدخلها للنموذج
y (Target) → النتيجة التي نريد من النموذج التنبؤ بها

ماذا يفعل؟

 df اسمه DataFrame  يأخذ
يحذف عمودين:

PriorityScore
PriorityLevel

 X  ويضع بقية الأعمدة في متغير اسمه 

 لماذا نحذف هذه الأعمدة؟
لأن:

PriorityScore هو الهدف (Target)
PriorityLevel غالبًا مشتق من PriorityScore
(مثل: High / Medium / Low)

❗ إذا تركناهم داخل X:

النموذج سيغش ✅
سيتعلم الإجابة بدل التنبؤ بها
(يسمى هذا Data Leakage)

ماذا يمثل y؟

هو Target Variable
القيمة التي نريد من النموذج أن يتعلم التنبؤ بها

استبدال الاكواد من تقسيم البيانات الى حفظ النموذج بالاكواد التالية لتحسين النموذج باستخدام :
RandomForestRegressor

In [35]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

# 1. Scaling
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


# -----------------------------
# Linear Regression
# -----------------------------
lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_pred)

lr_r2 = r2_score(y_test, lr_pred)

print(f"Linear Regression MAE: {lr_mae:.4f}")
print(f"Linear Regression R²: {lr_r2 * 100:.2f}%")

print("####################")

# -----------------------------
# Random Forest
# -----------------------------
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)

rf_r2 = r2_score(y_test, rf_pred)

print(f"Random Forest MAE: {rf_mae:.4f}")
print(f"Random Forest R²: {rf_r2 * 100:.2f}%")

Linear Regression MAE: 0.0000
Linear Regression R²: 100.00%
####################
Random Forest MAE: 0.0099
Random Forest R²: 99.04%


In [38]:
print("Linear Regression coefficients:")
print(lr_model.coef_)

print("\nIntercept:")
print(lr_model.intercept_)

print("\nTest R²:")
print(r2_score(y_test, lr_pred))

print("\nTest MAE:")
print(mean_absolute_error(y_test, lr_pred))

Linear Regression coefficients:
[0.4  0.3  0.1  0.08]

Intercept:
0.05999999999999972

Test R²:
1.0

Test MAE:
1.5334955527634976e-16


“حقق 
Linear Regression 
أقل خطأ لأنه مناسب للعلاقات الخطية المستخدمة في حساب 
PriorityScore، 
بينما أظهر 
Random Forest 
قدرة جيدة على فهم العلاقات المعقدة مع فرق بسيط في الدقة.”

“تم إنشاء PriorityScore 
باستخدام معادلة تعتمد على الخصائص الاجتماعية والاقتصادية، ثم تم تدريب النموذج لتعلم هذا السلوك وتوقع الأولويات تلقائيًا.”

### R² Score هي Accuracy
 Regression لكن لا تقال بشكل مباشر في 

In [36]:
import pandas as pd
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

importance.sort_values(by="Importance", ascending=False)


,Feature,Importance
0,SpecialCase,0.698476
1,Income,0.262030
3,Housing,0.025320
2,FamilyMembers,0.014174


⚠️ لماذا Region تأثيره شبه صفر؟

لأن:

البيانات غالبًا غير مرتبطة فعليًا بالمنطقة
أو التوزيع متشابه بين المناطق

🎯 وهذا طبيعي جدًا

ولا يعتبر خطأ

In [37]:

import pickle

with open("../models/priority_model.pkl", "wb") as f:
    pickle.dump(lr_model, f)

with open("../models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

قمنا بحفظ : 
1- model
2- scaler

لنستخدمهم في الخطوة التالية وهي : 
بناء API 